[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_1_2/04_tag_1_2_train_valid_test_kreuzvalidierung.ipynb)

# Tag 1-2 - 04 Training, Validierung, Test und Kreuzvalidierung

Dieses Notebook zeigt Training-, Validierungs-, CV- und Testfehler für jeden Polynomgrad. Die jeweils besten Punkte werden markiert, damit sichtbar wird, wenn ein Kriterium in die falsche Richtung führt.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
print('Setup abgeschlossen.')


In [ ]:
n = 90
sqm = np.sort(rng.uniform(25, 150, n))
x = (sqm - 85) / 40
true_rent = 850 + 420 * x + 140 * x**2 - 90 * x**3
rent = true_rent + rng.normal(0, 75, n)

indices = rng.permutation(n)
train_end = int(0.70 * n)
valid_end = int(0.80 * n)
train_idx = indices[:train_end]
valid_idx = indices[train_end:valid_end]
test_idx = indices[valid_end:]
train_valid_idx = indices[:valid_end]

display(pd.Series({'Training': len(train_idx), 'Validierung': len(valid_idx), 'Test': len(test_idx)}).to_frame('Samples'))


In [ ]:
def design(x_values, degree):
    return np.vstack([np.asarray(x_values) ** d for d in range(degree + 1)]).T

def fit_poly(x_train, y_train, degree):
    return np.linalg.lstsq(design(x_train, degree), y_train, rcond=None)[0]

def predict_poly(x_values, weights):
    return design(x_values, len(weights) - 1) @ weights

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

def kfold_rmse(x_values, y_values, degree, k=5):
    ids = np.arange(len(x_values))
    local_rng = np.random.default_rng(123)
    local_rng.shuffle(ids)
    folds = np.array_split(ids, k)
    scores = []
    for fold_id in range(k):
        val_ids = folds[fold_id]
        tr_ids = np.concatenate([folds[j] for j in range(k) if j != fold_id])
        weights = fit_poly(x_values[tr_ids], y_values[tr_ids], degree)
        scores.append(rmse(y_values[val_ids], predict_poly(x_values[val_ids], weights)))
    return float(np.mean(scores)), float(np.std(scores))


In [ ]:
rows = []
for degree in range(1, 19):
    weights = fit_poly(x[train_idx], rent[train_idx], degree)
    cv_mean, cv_std = kfold_rmse(x[train_valid_idx], rent[train_valid_idx], degree)
    rows.append({
        'Polynomgrad': degree,
        'Train RMSE': rmse(rent[train_idx], predict_poly(x[train_idx], weights)),
        'Valid RMSE': rmse(rent[valid_idx], predict_poly(x[valid_idx], weights)),
        'CV RMSE': cv_mean,
        'CV Std': cv_std,
        'Test RMSE': rmse(rent[test_idx], predict_poly(x[test_idx], weights)),
    })
tuning_df = pd.DataFrame(rows)
display(tuning_df.round(1))

best = {
    'Train RMSE': int(tuning_df.loc[tuning_df['Train RMSE'].idxmin(), 'Polynomgrad']),
    'Valid RMSE': int(tuning_df.loc[tuning_df['Valid RMSE'].idxmin(), 'Polynomgrad']),
    'CV RMSE': int(tuning_df.loc[tuning_df['CV RMSE'].idxmin(), 'Polynomgrad']),
    'Test RMSE': int(tuning_df.loc[tuning_df['Test RMSE'].idxmin(), 'Polynomgrad']),
}
display(pd.Series(best, name='bester Polynomgrad').to_frame())


In [ ]:
colors = {'Train RMSE': '#4c78a8', 'Valid RMSE': '#f58518', 'CV RMSE': '#54a24b', 'Test RMSE': '#e45756'}
for metric, color in colors.items():
    plt.plot(tuning_df['Polynomgrad'], tuning_df[metric], marker='o', color=color, label=f'{metric} (best {best[metric]})')
    y_best = tuning_df.loc[tuning_df['Polynomgrad'] == best[metric], metric].iloc[0]
    plt.scatter([best[metric]], [y_best], s=130, color=color, edgecolor='black', zorder=5)
plt.ylim(40, 260)
plt.xlabel('Polynomgrad')
plt.ylabel('RMSE in EUR')
plt.title('Train, Validierung, CV und Test pro Polynomgrad')
plt.legend()
plt.show()


In [ ]:
def degree_comparison(degree=4):
    weights = fit_poly(x[train_idx], rent[train_idx], degree)
    train_rmse = rmse(rent[train_idx], predict_poly(x[train_idx], weights))
    valid_rmse = rmse(rent[valid_idx], predict_poly(x[valid_idx], weights))
    test_rmse = rmse(rent[test_idx], predict_poly(x[test_idx], weights))

    grid_sqm = np.linspace(25, 150, 400)
    grid_x = (grid_sqm - 85) / 40
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].scatter(sqm[train_idx], rent[train_idx], alpha=0.45, label='Training')
    axes[0].scatter(sqm[valid_idx], rent[valid_idx], alpha=0.9, label='Validierung')
    axes[0].scatter(sqm[test_idx], rent[test_idx], alpha=0.9, label='Test')
    axes[0].plot(grid_sqm, 850 + 420 * grid_x + 140 * grid_x**2 - 90 * grid_x**3, color='green', linewidth=2, label='wahrer Zusammenhang')
    axes[0].plot(grid_sqm, predict_poly(grid_x, weights), color='black', linewidth=2, label=f'Grad {degree}')
    axes[0].set_ylim(250, 2200)
    axes[0].set_xlabel('Wohnfläche in qm')
    axes[0].set_ylabel('Miete in EUR')
    axes[0].set_title(f'Grad {degree}: Train RMSE={train_rmse:.1f} | Test RMSE={test_rmse:.1f}')
    axes[0].legend()

    axes[1].bar(['Train', 'Valid', 'Test'], [train_rmse, valid_rmse, test_rmse], color=['#4c78a8', '#f58518', '#e45756'])
    axes[1].set_ylabel('RMSE in EUR')
    axes[1].set_title('Fehler des ausgewählten Polynomgrads')
    plt.tight_layout()
    plt.show()

widgets.interact(degree_comparison, degree=widgets.IntSlider(value=4, min=1, max=18));
